# CS-781 Phase 3: Exp 4 — Layer-Wise Fine-Tuning (Model 3 GPT-NeoX)
Methodologically distinct Phase 3 Rhetorical Role Prediction submission using Layer-Wise Fine-Tuning.

In [ ]:
print('Using Kaggle native PyTorch environment for Layer-Wise Fine-Tuning.')


In [ ]:
import os, json, glob, zipfile, shutil, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, get_linear_schedule_with_warmup
from torch.optim import AdamW
from huggingface_hub import hf_hub_download, HfApi
from sklearn.metrics import f1_score, precision_score, recall_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

HF_TOKEN = 'YOUR_HF_TOKEN'
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
TARGET_REPO = 'U-241111/cs781-submissions'

# 1. Download Phase 3 Dataset
os.makedirs('/kaggle/working/data_p3', exist_ok=True)
p3_files = [
    'Phase_3_RR/Train/CL_train.json',
    'Phase_3_RR/Train/CL_dev.json',
    'Phase_3_RR/Train/IT_train.json',
    'Phase_3_RR/Train/IT_dev.json',
    'Phase_3_RR/Train/label_vocab.json',
    'Phase_3_RR/Test/CL_test.json',
    'Phase_3_RR/Test/IT_test.json',
]
for f in p3_files:
    fname = os.path.basename(f)
    loc = hf_hub_download(repo_id=HF_DATASET_REPO, filename=f, repo_type='dataset', token=HF_TOKEN)
    shutil.copyfile(loc, f'/kaggle/working/data_p3/{fname}')
print('Downloaded all Phase 3 files successfully!')


In [ ]:
# 2. Locate Phase 1 Model Checkpoint
model_ckpt_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'config.json' in files:
        model_ckpt_dir = root
        break
print('Model Checkpoint Path:', model_ckpt_dir, os.listdir(model_ckpt_dir))

tok = AutoTokenizer.from_pretrained(model_ckpt_dir)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token


In [ ]:
# 3. Context Window Samples & Dataset Construction
with open('/kaggle/working/data_p3/label_vocab.json') as f:
    label2id = json.load(f)
id2label = {v: k for k, v in label2id.items()}
num_labels = len(label2id)
print('Number of classification classes:', num_labels)

with open('/kaggle/working/data_p3/CL_train.json') as f: cl_train = json.load(f)
with open('/kaggle/working/data_p3/CL_dev.json') as f: cl_dev = json.load(f)
with open('/kaggle/working/data_p3/IT_train.json') as f: it_train = json.load(f)
with open('/kaggle/working/data_p3/IT_dev.json') as f: it_dev = json.load(f)

# Train on 100% of labeled data (CL_train + CL_dev + IT_train + IT_dev = 18,987 sentences)
train_docs = cl_train + cl_dev + it_train + it_dev
dev_docs = cl_dev + it_dev

def build_context_samples(doc_list, is_test=False):
    samples = []
    if isinstance(doc_list, dict):
        items = [{'id': k, 'text': v} for k, v in doc_list.items()]
    else:
        items = doc_list
        
    for doc in items:
        doc_id = doc['id']
        sents = doc['text']
        labels = doc.get('labels', [])
        n = len(sents)
        for i, s in enumerate(sents):
            pct = int((i / max(1, n - 1)) * 100)
            prev_s = sents[i-1] if i > 0 else "NONE"
            next_s = sents[i+1] if i < n - 1 else "NONE"
            
            prompt = f"[DOC_POS: {pct}%] [PREV] {prev_s} [CURR] {s} [NEXT] {next_s}"
            sample = {
                'doc_id': doc_id,
                'sent_idx': i,
                'text': prompt
            }
            if not is_test and labels:
                sample['label'] = label2id[labels[i]]
            samples.append(sample)
    return samples

train_samples = build_context_samples(train_docs)
dev_samples = build_context_samples(dev_docs)
print(f'Train samples: {len(train_samples)}, Dev samples: {len(dev_samples)}')

class RRDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=192, is_test=False):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_test = is_test
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        item = self.samples[idx]
        enc = self.tokenizer(
            item['text'],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        res = {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'doc_id': item['doc_id'],
            'sent_idx': item['sent_idx']
        }
        if not self.is_test:
            res['label'] = torch.tensor(item['label'], dtype=torch.long)
        return res


In [ ]:
# 4. Initialize Model 3 with Layer-Wise Fine-Tuning (Freezing Bottom 8 Layers)
model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt_dir,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)
model.config.pad_token_id = tok.pad_token_id

# Freeze bottom 8 transformer layers and embeddings, fine-tune top 8 layers + classification head
for name, p in model.named_parameters():
    is_top_layer = False
    for l_num in [8, 9, 10, 11, 12, 13, 14, 15]:
        if f'.{l_num}.' in name or f'layers.{l_num}' in name:
            is_top_layer = True
            break
    if is_top_layer or 'score' in name or 'classifier' in name:
        p.requires_grad = True
    else:
        p.requires_grad = False

model = model.to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Layer-wise adaptation configured! Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')


In [ ]:
# 5. Training Loop with Class-Weighted Cross-Entropy
train_loader = DataLoader(RRDataset(train_samples, tok, max_length=192), batch_size=32, shuffle=True)
optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4, weight_decay=0.01)
epochs = 5
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

class_weights = torch.tensor([1.0116, 3.4927, 1.3436, 1.4491, 1.5996, 2.6390, 1.3029, 2.0223, 7.8611, 1.7279, 1.0000, 2.4442, 2.5081], device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

print(f'Starting Layer-Wise Fine-Tuning training for {epochs} epochs ({total_steps} steps) on all 18,987 sentences...')
model.train()
for epoch in range(epochs):
    total_loss = 0
    t0 = time.time()
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attn_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
        loss = criterion(logits, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Time: {time.time()-t0:.1f}s')


In [ ]:
# 6. Validation and Test Inference
# 1. Validation on Dev Set
val_loader = DataLoader(RRDataset(dev_samples, tok, max_length=192), batch_size=32, shuffle=False)
model.eval()
all_preds = []
all_trues = []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attn_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_trues.extend(labels.cpu().numpy())

val_macro_f1 = f1_score(all_trues, all_preds, average='macro', zero_division=0)
val_macro_p = precision_score(all_trues, all_preds, average='macro', zero_division=0)
val_macro_r = recall_score(all_trues, all_preds, average='macro', zero_division=0)
print(f'>>> Validation Strict Macro-F1: {val_macro_f1:.4f} | Prec: {val_macro_p:.4f} | Rec: {val_macro_r:.4f} <<<')

for lbl, i in sorted(label2id.items(), key=lambda x: x[1]):
    c_f1 = f1_score([1 if t==i else 0 for t in all_trues], [1 if p==i else 0 for p in all_preds], zero_division=0)
    print(f'  Class {i:2d} ({lbl}): F1 = {c_f1:.4f}')

# 2. Test Set Inference (Exact 2,197 sentences across 10 documents)
with open('/kaggle/working/data_p3/CL_test.json') as f: cl_test = json.load(f)
with open('/kaggle/working/data_p3/IT_test.json') as f: it_test = json.load(f)

test_samples_cl = build_context_samples(cl_test, is_test=True)
test_samples_it = build_context_samples(it_test, is_test=True)
all_test_samples = test_samples_cl + test_samples_it
print(f'Total test samples: {len(all_test_samples)} (Expected: 2197)')

test_loader = DataLoader(RRDataset(all_test_samples, tok, max_length=192, is_test=True), batch_size=32, shuffle=False)
test_doc_preds = {d['id']: [] for d in [{'id': k} for k in list(cl_test.keys()) + list(it_test.keys())]}

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attn_mask = batch['attention_mask'].to(device)
        doc_ids = batch['doc_id']
        sent_indices = batch['sent_idx'].numpy()
        logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        
        for did, s_idx, p in zip(doc_ids, sent_indices, preds):
            test_doc_preds[did].append((int(s_idx), id2label[int(p)]))

# Sort by sentence index and extract ordered label list
final_predictions = {}
for did, sent_list in test_doc_preds.items():
    sent_list.sort(key=lambda x: x[0])
    final_predictions[did] = [lbl for _, lbl in sent_list]

# Verify exact sentence count per test document
for did, sents in final_predictions.items():
    expected_len = len(cl_test[did]) if did in cl_test else len(it_test[did])
    assert len(sents) == expected_len, f'Length mismatch in {did}: {len(sents)} vs expected {expected_len}'
print('✅ Exact length verification passed for all 10 test documents!')

# Package into CodaBench submission ZIP
pred_path = '/kaggle/working/prediction.json'
zip_path = '/kaggle/working/submission_phase3_exp4_layerwise.zip'

with open(pred_path, 'w', encoding='utf-8') as f:
    json.dump(final_predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname='prediction.json')

print(f'Created {zip_path} ({os.path.getsize(zip_path)/1024:.1f} KB)')

# Upload to Hugging Face
api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=zip_path,
    path_in_repo='phase3/submission_phase3_exp4_layerwise.zip',
    repo_id=TARGET_REPO,
    repo_type='dataset'
)
api.upload_file(
    path_or_fileobj=pred_path,
    path_in_repo='phase3/prediction_exp4_layerwise.json',
    repo_id=TARGET_REPO,
    repo_type='dataset'
)
print('Successfully uploaded Phase 3 submission to Hugging Face!')
